# Retrieving Relevant Chunks

In [6]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer

In [ ]:
# Load the previously generated embedded chunks
# Use pickle to read the embedded dataset
train_df_embedded = pd.read_pickle("train_df_chunked_embeddings.pkl")

In [14]:
# Check to see if the information is accurately pulled
# print(train_df_embedded.head(5))

# Since previously the chunk_id is a bit messy, converting it to a numerical udi would be cleaner
# This will be done by using the factorize functionality
train_df_embedded["chunk_id"] = range(1, len(train_df_embedded) + 1)

# Double check that the chunk_id has been adjusted properly
print(train_df_embedded.head(5))

                                              source  \
0  [Due to the success of deep learning to solvin...   
1  [Due to the success of deep learning to solvin...   
2  [Due to the success of deep learning to solvin...   
3  [Due to the success of deep learning to solvin...   
4  [Due to the success of deep learning to solvin...   

                                       source_labels  \
0  [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   
1  [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   
2  [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   
3  [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   
4  [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   

                                        rouge_scores   paper_id  \
0  [0.30188678746885, 0.37209301838831804, 0.6037...  SysEexbRb   
1  [0.30188678746885, 0.37209301838831804, 0.6037...  SysEexbRb   
2  [0.30188678746885, 0.37209301838831804, 0.6037...  SysEexbRb   
3  [0.30188678746885, 0.37209301838831804, 0.6037...  SysE

In [11]:
# Print the columns as a reminder
print("Old train_df_embedded Columns: ", train_df_embedded.columns)

# Rename the "source_chunks" to "chunk_text" to avoid misunderstandings
train_df_embedded = train_df_embedded.rename(columns = {'source_chunks': 'chunk_text'})

# Double check to make sure the column was properly renamed
print("Adjusted train_df_embedded Columns: ", train_df_embedded.columns)

Old train_df_embedded Columns:  Index(['source', 'source_labels', 'rouge_scores', 'paper_id', 'target', 'ic',
       'title', 'source_paragraph', 'target_paragraph', 'source_chunks',
       'chunk_no', 'embedding', 'chunk_id'],
      dtype='str')
Adjusted train_df_embedded Columns:  Index(['source', 'source_labels', 'rouge_scores', 'paper_id', 'target', 'ic',
       'title', 'source_paragraph', 'target_paragraph', 'chunk_text',
       'chunk_no', 'embedding', 'chunk_id'],
      dtype='str')


In [ ]:
# Create one matrix with the embedded column using vstack
# Reasoning: Computer can't efficiently search through a list of arrays so it needs to be combined into one large matrix
embedding_matrix = np.vstack(train_df_embedded["embedding"].values)

# Set up a search index to easily find relevant chunks
# Use nearestneighbor to capture the chunks that are the most relevant
# Cosine distance will be used since it conpares their direction, therefore meaning, instead of length
# The brute algorithm would be best suited since it works with cosine distance, the dataset is small enough, and ensures every chunk is checked
# n_neighbors can be mapped in the following function when retrieving chunks
vector_index = NearestNeighbors(metric = "cosine", algorithm = 'brute')

# Do not forget to fit the embedding matrix by using vector_index.fit()
vector_index.fit(embedding_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [15]:
# Create a function that retrieves the most relevant chunks
# Will be pulling the top 3 closest chunks in this function
# The query variable will be the input
# For later experimentation, top_k can be an adjustable variable

#################################################################
                        # top_k Parameters
#################################################################
top_k_3 = 3
top_k_4 = 4
top_k_5 = 5
#################################################################

#################################################################
                # Embedding Model Parameters
#################################################################
baai_model = "BAAI/bge-base-en-v1.5"
mini_lm_model = "all-MiniLM-L6-v2"
large_baai_model = "BAAI/bge-large-en-v1.5"
#################################################################

# Already defined previously but redefined again for safety
embedding_model = SentenceTransformer(baai_model)

def retrieve_chunks(query, top_k = top_k_3):

    # Make sure that the query is embedded first
    query_embedded = embedding_model.encode(
        [query], 
        convert_to_numpy = True,
        normalize_embeddings = True
    )

    # Pull the closest distance and index of the top 3 neighbors
    distance, indices = vector_index.kneighbors(
        query_embedded,
        n_neighbors = top_k)
    
    # Retrieve the matching rows by creating a copy and storing it as results
    results = train_df_embedded.iloc[indices[0]].copy()

    # Create a column to record the distance
    results["distance"] = distance[0]

    # Create another column to record the similarity score
    # Reminder that the similarity score is calculated by subtracting distance from 1
    # Similiarity measures how close a chunk matches the user input therefore higher is better
    results['similarity_score'] = 1 - results["distance"]

    # Return the columns
    return results[["chunk_id", "chunk_no", "chunk_text", "paper_id", "title", "distance", "similarity_score"]]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# Test to see if the retrieval works
# Recall from preprocessing section that the first record talked about how they explored the different analytical methods for the critical points in various neural networks

test_query = ("In what ways can we influence the critical points to better attune neural networks?")

# Try to retrieve the top 3 most relevant chunks
test_retrieval = retrieve_chunks(query = test_query, top_k = top_k_3)

# Print the chunks
print(test_retrieval.reset_index(drop = True))

# Test Query is successful

   chunk_id  chunk_no                                         chunk_text  \
0      1822         5  We hope our work will spur momentum towards th...   
1         5         5  namely, shallow linear networks, deep linear n...   
2      4351         3  that the proposed training techniques lead to ...   

     paper_id                                              title  distance  \
0  H1lma24tPB  Principled Weight Initialization for Hypernetw...  0.251710   
1   SysEexbRb  Critical Points of Linear Neural Networks: Ana...  0.254856   
2   SJa1Nk10b  Anytime Neural Network: a Versatile Trade-off ...  0.257588   

   similarity_score  
0          0.748290  
1          0.745144  
2          0.742412  
